# LettuceDetect baseline — Hallucination Detection in Tool Calling

Evaluates the off-the-shelf `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint on three test splits we built from ToolACE, each corresponding to one hallucination type:

- **Type 1** — *Hallucination*: answer contradicts tool output.
- **Type 2** — *Overgeneration*: answer adds facts not in tool output.
- **Type 3** — *Missing tool*: answer proposes an action requiring an unavailable tool.

For each split we report span-level (micro + macro F1) and example-level (P/R/F1) metrics.


In [ ]:

!pip install --upgrade pip

!pip install torch --index-url https://download.pytorch.org/whl/cu121

!pip install "transformers>=4.48.0" "accelerate>=0.26.0" pandas numpy tqdm ipywidgets

!pip install lettucedetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 55.4 MB/s  0:00:12:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 42.6 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 52.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 74.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 41.1 MB/s  0:00:12:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 62.7 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 53.8 MB/s  0:00:02:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 71.8 MB/s  0:00:00 eta 0:00:01
     ━━━

In [2]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import torch
from lettucedetect.models.inference import HallucinationDetector

os.makedirs("/app/results", exist_ok=True)
DATA_DIR = Path("/app/data")
MODEL_PATH = "KRLabsOrg/lettucedect-large-modernbert-en-v1"

TEST_FILES = {
    "Combined": DATA_DIR / "combined_test.jsonl",
    "Type 1 + clean": DATA_DIR / "type1_test_balanced.jsonl",
    "Type 2 + clean": DATA_DIR / "type2_test_balanced.jsonl",
    "Type 3 + clean": DATA_DIR / "type3_test_balanced.jsonl",
}

print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA: False | GPU: None


/home/serov_docker/miniconda/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _char_set(spans):
    chars = set()
    for s in spans:
        chars.update(range(int(s["start"]), int(s["end"])))
    return chars

def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample.get("hallucination_labels", []))
        pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap
        micro_pred += len(pred)
        micro_gold += len(gold)
        
        if not pred and not gold:
            macro_f1.append(1.0); continue
            
        p = overlap / len(pred) if pred else 0.0
        r = overlap / len(gold) if gold else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        macro_f1.append(f1)
        
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    f_ma = sum(macro_f1) / len(macro_f1) if macro_f1 else 0.0
    return {"P": p_mi, "R": r_mi, "F1": f_mi, "macro_F1": f_ma}

def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample.get("hallucination_labels") else 0
        pred = 1 if preds else 0
        if gold == 1 and pred == 1: tp += 1
        elif gold == 1 and pred == 0: fn += 1
        elif gold == 0 and pred == 1: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"P": p, "R": r, "F1": f1}

In [4]:
print("Загрузка LettuceDetect...")
detector = HallucinationDetector(method="transformer", model_path=MODEL_PATH)

def predict(sample):
    raw = detector.predict(
        context=[sample.get("context", "")],
        question=sample.get("query", ""),
        answer=sample.get("output", ""),
        output_format="spans",
    )
    return [
        {"start": int(s["start"]), "end": int(s["end"]),
         "text": s.get("text", sample["output"][int(s["start"]):int(s["end"])]),
         "confidence": float(s.get("confidence", 0.0))}
        for s in raw
    ]

results = []
for name, path in TEST_FILES.items():
    if not path.exists():
        print(f"[skip] {name}: {path} not found")
        continue
    
    samples = read_jsonl(path)
    preds = [predict(s) for s in tqdm(samples, desc=name)]
    
    sm = span_metrics(samples, preds)
    em = example_metrics(samples, preds)
    
    results.append({
        "Split": name, "N": len(samples),
        "Span P": round(sm["P"], 3), "Span R": round(sm["R"], 3), "Span F1": round(sm["F1"], 3), "Span macro F1": round(sm["macro_F1"], 3),
        "Ex P": round(em["P"], 3), "Ex R": round(em["R"], 3), "Ex F1": round(em["F1"], 3)
    })

with open("/app/results/lettuce_baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

df = pd.DataFrame(results)
display(df)
print("saved /app/results/lettuce_baseline_metrics.json")

Загрузка LettuceDetect...


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Type 1 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 2 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 3 + clean:   0%|          | 0/299 [00:00<?, ?it/s]

,Split,N,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
0,Combined,599,0.515,0.918,0.660,0.629,0.871,0.902,0.886
1,Type 1 + clean,300,0.077,0.669,0.137,0.440,0.661,0.780,0.716
2,Type 2 + clean,300,0.570,0.998,0.726,0.746,0.714,1.000,0.833
3,Type 3 + clean,299,0.461,0.847,0.597,0.671,0.697,0.926,0.795


saved /app/results/lettuce_baseline_metrics.json
